# Recurse

The Recurse R package (C. Bracis <i>et al.</i>, Ecography, 2018 : https://onlinelibrary.wiley.com/doi/abs/10.1111/ecog.03618) can be used to analyze animal trajectory data to look for returns to a previously visited area, i.e. revisitations. These revisits could be interesting ecologically for a number of reasons. For example, they could be used to identify nesting or denning sites or important resource locations such as water points.

The original method from the paper works by taking a circle of a user-specified radius and moving it along a set a relocations points. At each point, the number of trajectory segments entering and exiting the circle is counted to determine the number of revisitation. Therefore, each movement trajectory location has one visit for the piece of the trajectory centered on it plus additional visits that could occur before or after. 

Our implementation of the algorithm proposed by Bracis <i>et al.</i> is a little bit different than the one described in their paper. Indeed, we are counting the number of trajectory segments inside a <b>square</b> instead of a circle for each relocation points. The length of the sides of the square (in meters) must be specified by the user through the `resolution` parameter. Our method gives almost identical results to the one described in the paper, and is faster to execute due to our usage of affine transforms. Figure 1 shows the difference between our implementation and the one available in the R recurse package.

<div align="center">
<img src="https://i.ibb.co/Tc5rcFf/Screen-Shot-2022-10-24-at-7-28-03-PM.png" width=300 height=auto />
<br> Figure 1. Difference between the Recurse R package and Ecoscope for calculating the number of revisits. 
</div>

## Ecoscope

In [ ]:
ECOSCOPE_RAW = "https://raw.githubusercontent.com/wildlife-dynamics/ecoscope/master"

In [ ]:
import os
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point
from sklearn.cluster import KMeans

import ecoscope

ecoscope.init()

## Google Drive Setup

In [ ]:
output_dir = "Ecoscope-Outputs"

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive/", force_remount=True)
    output_dir = os.path.join("/content/drive/MyDrive/", output_dir)

os.makedirs(output_dir, exist_ok=True)

## Create Relocations

In [ ]:
ecoscope.io.download_file(
    f"{ECOSCOPE_RAW}/tests/sample_data/vector/movbank_data.csv",
    os.path.join(output_dir, "movbank_data.csv"),
)

df = pd.read_csv(os.path.join(output_dir, "movbank_data.csv"), index_col=0)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["location-long"], df["location-lat"]), crs=4326)

relocs = ecoscope.base.Relocations.from_gdf(gdf, groupby_col="individual-local-identifier", time_col="timestamp")
relocs_salif = relocs[relocs.groupby_col == "Salif Keita"].copy()

## Calculating and visualizing revisits

In [ ]:
relocs_salif["revisits"] = ecoscope.analysis.get_recursions(relocs_salif, resolution=400)
relocs_salif.explore(column="revisits", cmap="viridis")

## Clustering revisits

If important sites are not known a priori, they can be identified from the recursion analysis using clustering. For example, one might want to identify nests, dens, water holes, foraging patches, roosts, haulouts, etc. in a systematic way rather than using visual identification methods, which may not be feasible for large amounts of data. The identification of these sites may be the end goal itself, or a preliminary step in a larger analysis.

As an example, here we use K-Means clustering to cluster the (x,y) coordinates of the top 20% of locations by number of revisists. Please note that many other clustering methods are also available in the sklearn package : https://scikit-learn.org/stable/modules/clustering.html. 

In [ ]:
relocs_top20 = relocs_salif[relocs_salif["revisits"] >= np.percentile(relocs_salif["revisits"], 80)].copy()
relocs_top20.explore(column="revisits", cmap="viridis")

In [ ]:
utm_crs = relocs_top20.estimate_utm_crs()
relocs_top20.to_crs(utm_crs, inplace=True)
relocs_top20["utm_east"] = [relocs_top20["geometry"].iloc[i].x for i in range(len(relocs_top20))]
relocs_top20["utm_north"] = [relocs_top20["geometry"].iloc[i].y for i in range(len(relocs_top20))]

kmeans = KMeans(n_clusters=6)
relocs_top20["cluster"] = kmeans.fit_predict(relocs_top20[["utm_east", "utm_north"]])
relocs_top20.explore()

In [ ]:
cluster_idx = np.unique(relocs_top20["cluster"])
cluster_centers = [Point(cluster_center) for cluster_center in list(kmeans.cluster_centers_)]
cluster_radius = [
    np.max(relocs_top20[relocs_top20.cluster == idx].geometry.distance(cluster))
    for idx, cluster in zip(cluster_idx, cluster_centers)
]
cluster_gdfs = [gpd.GeoDataFrame(geometry=[cluster], crs=utm_crs) for cluster in cluster_centers]
for i in range(6):
    cluster_gdfs[i]["geometry"] = cluster_gdfs[i].buffer(cluster_radius[i])

In [ ]:
m = ecoscope.mapping.EcoMap(width=800, height=600)
m.zoom_to_gdf(relocs_top20)
for cluster_gdf in cluster_gdfs:
    m.add_gdf(cluster_gdf)
m